# 1 Data Collection and Preparation

In order to structure and prepare the data set it is important to be sure what questions the data set should answer in the end.
The main questions are:
1. where is demand?
2. when is demand?
3. which factors determine demand?
4. how good can demand be projected?
5. topic charging (when, how strong, only private charging hubs, public infrastructure, ...)

To answer especially the forth question it is important to also consider which spatial (census tracts / hexagons) or time-based (hourly, 4-hourly) distribution works the best.

Final consulting should include in which districts to enter market at first (strategic), how many cars should be available when and where (tactical) and how to charge and reposition vehicles (operational).

- *1.1 Overview on dataset*
- *1.2 Syntactical Cleaning*
- *1.3 Missing Value Strategy*
- *1.4 Semantic Validation and Outlier Handling*
- *1.5 External Data Enrichment*
- *1.6 Spatial and Temporal Aggregation*
- *1.7 Result: Starting Point for further analysis*

Section 1.1 first provides a general overview of the raw dataset, including its columns, missing values, and basic structure. At the end of Section 1.1, the logic behind the overall preparation structure is explained in more detail. This is useful because the later cleaning and aggregation steps should not be arbitrary, but directly follow from the business questions and the intended final structure of the modeling dataset.


## 1.1 Overview on data set

To have a first glance into the data, some initial information about data types of each column and missing values per column should be considered, to maybe alright shorten the working set before going deeper into data preparation.

In [2]:
# the whole dataset until 4/30/26 is loaded and locally imported into the data folder, but it gets ignored by gitignore
# at first, only use the first 100.000 entries (of the total 50 million) to reduce computational effort
import pandas as pd

df = pd.read_csv(
    "../data/Taxi_Trips_(2024-)_20260502.csv",
    nrows=100_000
)

df_full = pd.read_csv(
    "../data/Taxi_Trips_(2024-)_20260502.csv"
)

df_full.head()

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,6c54cdb22905b181c23527e74c2b653a503107b7,db757f6c1157d9f81e266396132cd641837c189b803c52...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.407,"14,35",NaN,NaN,76.0,22.0,...,"$5,00","$48,01",Credit Card,Flash Cab,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,92276062","-87,699155343",POINT (-87.6991553432 41.9227606205)
1,6cd5d74b7ec8d7d230387900d72d74326dfb31de,f509c57d2f0b196f54d9b751ce2a1b6e956f378841b1e8...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.701,"19,14",1.703198e+10,1.703132e+10,76.0,32.0,...,"$0,00","$60,50",Credit Card,City Service,"41,97907082","-87,903039661",POINT (-87.9030396611 41.9790708201),"41,884987192","-87,620992913",POINT (-87.6209929134 41.8849871918)
2,d36a90d961ed60add20ab3a4a7a8ca98bd2bf929,4136627ef25b9fad79910c55679c02d8e1f2a42925d29c...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.260,"10,2",NaN,NaN,76.0,14.0,...,"$4,00","$31,25",Cash,Transit Administrative Center Inc,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,968069","-87,721559063",POINT (-87.7215590627 41.968069)
3,cb1fed57e7eeea1db20758256b754201f5e13e07,c9bda81f1aaad786527911c983b6be11f28c6f46469e1a...,04/01/2026 12:00:00 AM,04/01/2026 12:30:00 AM,1.608,"18,4",NaN,NaN,76.0,32.0,...,"$5,00","$50,50",Credit Card,Taxicab Insurance Agency Llc,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841)
4,d3c33c166fdbca45d040772d18bd1ae1dc669ade,f71223a469d78a2f65a090adc9d4fb5ae08dfc6694c650...,04/01/2026 12:00:00 AM,04/01/2026 12:00:00 AM,422.000,"0,98",NaN,NaN,32.0,28.0,...,"$0,00","$6,00",Cash,Chicago Independents,"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841),"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383)


In [3]:
# just to take a look at the data types it might be beneficial to print df.info()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Trip ID                     100000 non-null  str    
 1   Taxi ID                     100000 non-null  str    
 2   Trip Start Timestamp        100000 non-null  str    
 3   Trip End Timestamp          99998 non-null   str    
 4   Trip Seconds                99988 non-null   float64
 5   Trip Miles                  100000 non-null  str    
 6   Pickup Census Tract         45638 non-null   float64
 7   Dropoff Census Tract        44444 non-null   float64
 8   Pickup Community Area       97473 non-null   float64
 9   Dropoff Community Area      91399 non-null   float64
 10  Fare                        99892 non-null   str    
 11  Tips                        99892 non-null   str    
 12  Tolls                       99892 non-null   str    
 13  Extras                    

In [4]:
# in df.info() we see some missing values
# we test that based on the real data set to see if those structures really influence the whole data set
missing_overview = pd.DataFrame({
    "missing_count": df_full.isna().sum(),
    "missing_percent": df_full.isna().mean() * 100
})

missing_overview = missing_overview.sort_values(
    by="missing_count",
    ascending=False
)

missing_overview

,missing_count,missing_percent
Dropoff Census Tract,8435004,56.982301
Pickup Census Tract,8237431,55.647605
Dropoff Community Area,1320108,8.917932
Dropoff Centroid Longitude,1241359,8.385947
Dropoff Centroid Location,1241359,8.385947
Dropoff Centroid Latitude,1241359,8.385947
Pickup Community Area,413385,2.792604
Pickup Centroid Location,405819,2.741493
Pickup Centroid Latitude,405819,2.741493
Pickup Centroid Longitude,405819,2.741493


After assessing missing values and reviewing the available column information, the dataset structure is summarized to provide a basis for deciding which columns should be retained or excluded for the initial analysis.

| Column | Meaning | Relevance | Missing Values (%) |
|---|---|---|---:|
| `Trip ID` | Unique identifier of each trip | Technical key, e.g. for detecting duplicates | 0.00 |
| `Taxi ID` | Anonymized identifier of the taxi | Vehicle-level analysis, idle time, vehicle utilization | 0.00 |
| `Trip Start Timestamp` | Start time of the trip, rounded to 15-minute intervals | Demand analysis by time, weekday, and calendar period | 0.00 |
| `Trip End Timestamp` | End time of the trip, rounded to 15-minute intervals | Trip duration validation, temporal pattern analysis | 0.00 |
| `Trip Seconds` | Duration of the trip in seconds | Travel time, speed calculation, outlier detection | 0.02 |
| `Trip Miles` | Distance of the trip in miles | Trip length, efficiency, cost-related analysis | 0.00 |
| `Pickup Census Tract` | Census tract where the trip starts | Fine-grained spatial demand analysis | 55.65 |
| `Dropoff Census Tract` | Census tract where the trip ends | Destination areas, mobility flows | 56.98 |
| `Pickup Community Area` | Community area of the pickup location | Coarser spatial demand analysis | 2.79 |
| `Dropoff Community Area` | Community area of the dropoff location | Destination patterns, origin-destination analysis | 8.92 |
| `Fare` | Base fare of the trip | Revenue analysis, price level | 0.21 |
| `Tips` | Tip amount | Payment and service behavior; less central for demand prediction | 0.21 |
| `Tolls` | Toll costs | Cost component, rather a control variable | 0.21 |
| `Extras` | Additional charges | Additional costs, e.g. airport-related fees | 0.21 |
| `Trip Total` | Total amount paid for the trip | Revenue per trip | 0.21 |
| `Payment Type` | Payment method, e.g. cash or credit card | User behavior, data quality checks | 0.00 |
| `Company` | Taxi company operating the trip | Provider structure, possible filtering variable | 0.00 |
| `Pickup Centroid Latitude` | Latitude of the pickup area centroid | Mapping, H3 hexagons, spatial modeling | 2.74 |
| `Pickup Centroid Longitude` | Longitude of the pickup area centroid | Mapping, H3 hexagons, spatial modeling | 2.74 |
| `Pickup Centroid Location` | Point coordinate of the pickup area centroid | GIS processing; redundant if latitude and longitude are used | 2.74 |
| `Dropoff Centroid Latitude` | Latitude of the dropoff area centroid | Destination hotspots, spatial destination analysis | 8.39 |
| `Dropoff Centroid Longitude` | Longitude of the dropoff area centroid | Destination hotspots, spatial destination analysis | 8.39 |
| `Dropoff Centroid Location` | Point coordinate of the dropoff area centroid | GIS processing; redundant if latitude and longitude are used | 8.39 |

### Conceptual Considerations for Dataset Preparation

The raw taxi trip dataset consists of individual trip records. However, not every column is equally relevant for answering the business questions of the project. The final objective is not only to describe individual taxi trips, but to support a future ride-hailing fleet operator in understanding **where**, **when**, and under which conditions taxi demand occurs.

Therefore, the main analytical goal is to transform the transactional trip-level data into aggregated demand datasets. These datasets should allow us to answer questions such as:

- In which areas does taxi demand occur most frequently?
- At which times of day or days of the week is demand highest?
- How does demand vary across different spatial and temporal resolutions?
- Which resolution is most useful for operational fleet planning?
- How can external factors, such as weather, improve the understanding and prediction of demand?

The central target variable for the analysis is therefore:

**Demand count = number of taxi pickups per location unit and time bucket**

For example:

| Time Bucket | Location Unit | Demand Count |
|---|---|---:|
| 2024-01-01 08:00 | Community Area 32 | 57 |

This structure is much more relevant for the business problem than analyzing every taxi trip in isolation, because it directly reflects the amount of expected demand in a specific area at a specific time.

---

### Spatial and Temporal Aggregation Logic

A key question is how the final analytical dataset should be structured. Since the assignment requires the analysis of different spatial and temporal resolutions, several combinations should be considered.

#### Temporal resolution

Potential time buckets include:

- 15-minute intervals
- hourly intervals
- 4-hour intervals

A finer temporal resolution, such as 15 minutes, provides more detailed operational insights but may also lead to more volatile demand values. A coarser resolution, such as 4 hours, is more stable but less precise for short-term operational decisions.

#### Spatial resolution

Potential spatial units include:

- Census Tracts
- Community Areas
- H3 hexagons based on centroid coordinates

Community Areas offer a robust and interpretable spatial level with relatively few missing values. Census Tracts provide a more detailed spatial resolution but contain a high share of missing values. H3 hexagons allow for a flexible and standardized spatial grid, which can be adjusted depending on the required level of detail.

Combining three temporal and three spatial resolutions could theoretically lead to **nine aggregated datasets**:

| Spatial Resolution | Temporal Resolution |
|---|---|
| Community Area | 15 minutes |
| Community Area | 1 hour |
| Community Area | 4 hours |
| Census Tract | 15 minutes |
| Census Tract | 1 hour |
| Census Tract | 4 hours |
| H3 Hexagons | 15 minutes |
| H3 Hexagons | 1 hour |
| H3 Hexagons | 4 hours |

However, it may not be efficient to start with all nine datasets immediately. A more practical approach is to begin with one robust baseline dataset, for example **Community Area × 1 hour**, and then extend the analysis step by step.

---

### Handling Missing Spatial Data

Missing values should not be removed blindly. This is especially important for spatial variables such as Census Tracts. If more than half of the Census Tract values are missing, simply dropping these rows would strongly reduce the observed demand count and potentially distort the analysis.

For example, if only trips with available Census Tract information are retained, the resulting demand counts no longer represent total taxi demand. Instead, they only represent the subset of trips for which Census Tract information is available.

Therefore, the treatment of missing values should depend on the selected spatial resolution:

- For **Community Area-based datasets**, only rows with missing pickup community area need to be excluded.
- For **Census Tract-based datasets**, only rows with available pickup census tract can be used, but the limited data coverage must be clearly documented.
- For **H3-based datasets**, rows with missing pickup latitude or longitude need to be excluded.

A possible adjustment would be to scale observed Census Tract counts upward based on the share of available Census Tract values. However, this would assume that missing Census Tract values are randomly distributed. Since this assumption is likely uncertain, such an adjustment should be treated cautiously and not used as the main approach.

A more robust strategy is to use Community Areas and H3 hexagons as the main spatial levels and treat Census Tracts mainly as a sensitivity analysis.

---

### Transactional Data vs. Aggregated Data

It is important to distinguish between the raw transactional dataset and the final aggregated modeling dataset.

#### Transactional dataset

The transactional dataset contains one row per taxi trip. It is useful for:

- data quality checks
- duplicate detection
- outlier detection
- calculating trip durations and distances
- analyzing idle times between trips using the anonymized taxi ID
- calculating additional aggregated features later

Typical columns in this dataset include:

- trip start and end timestamp
- taxi ID
- trip seconds
- trip miles
- pickup and dropoff location information
- fare and trip total
- company and payment type

This dataset should be kept as a cleaned trip-level version because later analyses may require going back to the individual trip level.

#### Aggregated modeling dataset

The aggregated dataset contains one row per combination of time bucket and location unit. It is the main basis for descriptive demand analysis and predictive modeling.

A typical structure is:

| Time Bucket | Location Unit | Demand Count | Average Trip Miles | Average Trip Seconds | Average Trip Total | Weather Features |
|---|---|---:|---:|---:|---:|---|

Columns such as `Company`, `Payment Type`, `Trip ID`, or `Extras` are not directly useful in the aggregated dataset unless they are transformed into meaningful aggregate features, such as the number of different companies active in an area or the average fare per trip.

The aggregated dataset does not need to remain directly connected to individual `Trip ID`s. If additional features are needed later, they can be recalculated from the cleaned transactional dataset.

---

### Column Selection for the Initial Analysis

For the first analysis, the focus should be on columns that help answer the core demand question:

**How many taxi pickups occur in a specific area at a specific time?**

Therefore, the initial dataset should mainly include variables related to time, pickup location, basic trip characteristics, and possibly revenue.

The most important columns are:

- `Trip Start Timestamp`
- `Pickup Community Area`
- `Pickup Census Tract`
- `Pickup Centroid Latitude`
- `Pickup Centroid Longitude`
- `Trip Seconds`
- `Trip Miles`
- `Trip Total`

These columns are sufficient to create the first demand datasets and to describe demand patterns across time and space.

Some columns should be kept in the cleaned trip-level dataset, but do not need to be part of the first aggregated modeling dataset. This includes:

- `Taxi ID`, because it may be useful for idle time and vehicle utilization.
- `Trip End Timestamp`, because it is needed to validate trip duration and calculate idle time.
- `Dropoff Community Area`, `Dropoff Census Tract`, and dropoff coordinates, because they are useful for later origin-destination analysis.

Other columns are less relevant for the first demand model:

- `Trip ID`
- `Payment Type`
- `Company`
- `Fare`
- `Tips`
- `Tolls`
- `Extras`
- `Pickup Centroid Location`
- `Dropoff Centroid Location`

These columns should not necessarily be deleted from the raw data. However, they are not required for the first demand aggregation. Especially `Pickup Centroid Location` and `Dropoff Centroid Location` are redundant if latitude and longitude are already available.

---


Based on the information known to this point in time, a suitable structure to work on task 1 can be created. This structure includes the following points.

### 1.2 Syntactical Cleaning

This step focuses on technical issues in the data format.

The goal is to make the dataset technically usable before interpreting the values.

---

### 1.3 Missing Value Strategy

Before checking the semantic plausibility of individual values, missing values should first be analyzed. This is important because the later demand dataset is created by aggregating trips across spatial and temporal units.

Missing values should not be handled globally. Instead, the required columns depend on the intended spatial resolution.

For pickup-based demand, rows need a valid pickup time and a valid pickup location at the selected spatial level. Dropoff information is useful for later OD-analysis or repositioning, but it is not required for counting pickup demand.

The strategy should therefore be specific to each spatial resolution:

| Dataset Type | Required Location Information |
|---|---|
| Community Area dataset | `Pickup Community Area` |
| Census Tract dataset | `Pickup Census Tract` |
| H3 dataset | `Pickup Centroid Latitude` and `Pickup Centroid Longitude` |

This means that the number of usable rows differs across datasets. These differences should be documented clearly, especially for Census Tracts, where missing values are substantial.

---

### 1.4 Semantic Validation and Outlier Handling

After the missing value structure has been analyzed, the available values should be checked for plausibility.

Not every unusual value should be removed automatically. Some values may be rare but valid. For example, a short trip with a high price may look suspicious, but could still be possible due to congestion, waiting time, or additional charges.

Therefore, outliers should first be inspected and then either removed, capped, or flagged depending on the case. For the first demand analysis, the most important requirement is that the pickup time and the selected pickup location are valid.

---

### 1.5 External Data Enrichment

Weather data should be added after the time buckets have been created.

For hourly demand datasets, hourly weather data can be merged directly by timestamp. For 4-hour buckets, weather variables should be aggregated accordingly. 

Weather data should not be treated as a separate analysis only. It should become part of the final demand dataset, because weather may help explain why demand changes across time.

Furthermore other external data sources should be considered at this point.

---

### 1.6 Spatial and Temporal Aggregation

After cleaning and feature creation, the trip-level data can be aggregated.

The basic aggregation is:

**Group by time bucket and location unit, then count the number of pickups.**

This creates the main target variable:

`demand_count`

Additional aggregate variables can also be calculated, such as:

- average trip duration
- average trip distance
- average trip total
- number of unique taxis
- number of dropoffs
- average idle time, if calculated beforehand

---

### 1.7 Result: Starting Point for further analysis

The starting point is:

**Community Area × 1 hour**

This dataset is a suitable baseline because it has relatively few missing values, is easy to explain, and directly supports the main business question.

After this baseline is created, further versions can be tested e.g. on:

- Community Area × 4 hours
- H3 × 1 hour
- H3 × 4 hours
- Census Tract datasets as sensitivity checks

## 1.2 Syntactical Cleaning

This chapter focuses on technical data cleaning steps that are necessary before the dataset can be analyzed. The goal is to ensure that all columns have consistent formats and can be processed correctly.

The main steps include:

- standardizing column names
- converting timestamps into datetime format
- converting numeric and monetary columns into proper numeric types
- checking for duplicate rows

This step does not yet evaluate whether values are realistic in a business sense. It only ensures that the dataset is technically clean and usable.

For easier handling, the column names get standardized.

In [5]:
# Standardize column names for easier handling

df_full.columns = (
    df_full.columns
    .str.strip()                  # remove leading/trailing spaces
    .str.lower()                  # convert to lowercase
    .str.replace(" ", "_")         # replace spaces with underscores
)

df_full.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid__location'],
      dtype='str')

The trip start and end timestamps are currently stored as raw values. Converting them into datetime format is necessary to extract time-based features such as hour, weekday, month, and different time buckets for later demand aggregation.

In [6]:
timestamp_cols = [
    "trip_start_timestamp",
    "trip_end_timestamp"
]

for col in timestamp_cols:
    df_full[col] = pd.to_datetime(df_full[col], errors="coerce")

df_full[timestamp_cols].dtypes

C:\Users\Georg\AppData\Local\Temp\ipykernel_29140\1246117402.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_full[col] = pd.to_datetime(df_full[col], errors="coerce")
C:\Users\Georg\AppData\Local\Temp\ipykernel_29140\1246117402.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_full[col] = pd.to_datetime(df_full[col], errors="coerce")


trip_start_timestamp    datetime64[us]
trip_end_timestamp      datetime64[us]
dtype: object

Several columns contain numeric values such as trip duration, trip distance, coordinates, and trip costs. These columns need to be stored as numeric data types to enable filtering, aggregation, outlier detection, and later modeling. Even if the values appear numeric in the raw CSV file, they may initially be imported as strings and should therefore be converted explicitly.

In [7]:
# Convert trip_seconds: "1.407" -> 1407
df_full["trip_seconds"] = (
    df_full["trip_seconds"]
    .astype("string")
    .str.replace(".", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Convert decimal / monetary columns: "14,35" -> 14.35 and "$48,01" -> 48.01
num_cols = [
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude"
]

for col in num_cols:
    df_full[col] = (
        df_full[col]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

# Treat IDs / categories as strings
id_cols = [
    "trip_id",
    "taxi_id",
    "pickup_census_tract",
    "dropoff_census_tract",
    "pickup_community_area",
    "dropoff_community_area",
    "payment_type",
    "company"
]

for col in id_cols:
    df_full[col] = df_full[col].astype("string")

# Check result
df_full.dtypes

trip_id                               string
taxi_id                               string
trip_start_timestamp          datetime64[us]
trip_end_timestamp            datetime64[us]
trip_seconds                           Int64
trip_miles                           Float64
pickup_census_tract                   string
dropoff_census_tract                  string
pickup_community_area                 string
dropoff_community_area                string
fare                                 Float64
tips                                 Float64
tolls                                Float64
extras                               Float64
trip_total                           Float64
payment_type                          string
company                               string
pickup_centroid_latitude             Float64
pickup_centroid_longitude            Float64
pickup_centroid_location                 str
dropoff_centroid_latitude            Float64
dropoff_centroid_longitude           Float64
dropoff_ce

Types seem correct.

Duplicate records can distort demand counts because the same trip would be counted more than once. Therefore, the dataset is checked for fully duplicated rows and, additionally, for duplicated trip IDs.

In [8]:
# Check fully duplicated rows
n_full_duplicates = df_full.duplicated().sum()

print(f"Number of fully duplicated rows: {n_full_duplicates}")

Number of fully duplicated rows: 0


In [9]:
# Check duplicated Trip IDs
n_duplicate_trip_ids = df_full["trip_id"].duplicated().sum()

print(f"Number of duplicated trip IDs: {n_duplicate_trip_ids}")

Number of duplicated trip IDs: 0


After those initial checks, the data set is syntactical clean for the upcoming purposes.

## 1.3 Missing Value Strategy

After the dataset has been technically cleaned, the next step is to analyze missing values. This is especially important because the later demand dataset is created by aggregating trips across spatial and temporal units. Therefore, missing values must not be handled globally, but depending on the intended aggregation level.

In this dataset, missing values are not equally distributed across all spatial columns. Census Tract information has a very high share of missing values, while Community Area and centroid coordinates are much more complete. Therefore, dropping all rows with missing values would remove a large part of the dataset and could strongly distort the observed demand patterns.

The main checks in this section include:

- copy the number and percentage of missing values per column from 1.1
- comparing missing values across spatial columns:
  - `pickup_census_tract`
  - `pickup_community_area`
  - `pickup_centroid_latitude`
  - `pickup_centroid_longitude`
- checking whether missing pickup information affects certain times, areas, companies, or trip characteristics more strongly
- deciding which spatial columns are reliable enough for the main analysis
- defining separate filtering rules for each later spatial dataset:
  - Community Area dataset requires `pickup_community_area`
  - Census Tract dataset requires `pickup_census_tract`
  - H3 / hexagon dataset requires `pickup_centroid_latitude` and `pickup_centroid_longitude`
- avoiding a global drop of all missing values
- documenting why Census Tracts are only used carefully, for example as a sensitivity analysis
- documenting why Community Area × time is used as the first practical aggregation level

The central principle is that missing values are handled based on the analytical purpose of each dataset. For the initial demand model, trips only need a valid pickup time and pickup location at the selected spatial resolution. Other missing values, such as dropoff information or payment details, should not automatically lead to row removal because they are not required for counting pickup demand.

The actual removal of rows with missing values is therefore postponed until the corresponding aggregation dataset is created.

In [10]:
# Check missing values overview again after type conversions
missing_overview = pd.DataFrame({
    "missing_count": df_full.isna().sum(),
    "missing_percent": df_full.isna().mean() * 100
})

missing_overview = missing_overview.sort_values(
    by="missing_count",
    ascending=False
)

missing_overview

,missing_count,missing_percent
dropoff_census_tract,8435004,56.982301
pickup_census_tract,8237431,55.647605
dropoff_community_area,1320108,8.917932
dropoff_centroid_longitude,1241359,8.385947
dropoff_centroid__location,1241359,8.385947
dropoff_centroid_latitude,1241359,8.385947
pickup_community_area,413385,2.792604
pickup_centroid_location,405819,2.741493
pickup_centroid_latitude,405819,2.741493
pickup_centroid_longitude,405819,2.741493


After syntactical cleaning, most missing value counts remain unchanged. Only a few numeric columns such as `trip_total`, `fare`, `extras`, `tolls`, and `trip_miles` show a small increase in missing values, likely caused by non-convertible values being coerced to `NaN`. The spatial missing value pattern remains unchanged.

| Column       | before | after   | difference  |
| ------------ | -----: | ------: | ----------: |
| `trip_total` | 31,160 |  32,943 |      +1,783 |
| `fare`       | 31,160 |  32,894 |      +1,734 |
| `extras`     | 31,160 |  31,198 |         +38 |
| `tolls`      | 31,160 |  31,174 |         +14 |
| `trip_miles` |    123 |     139 |         +16 |


The missing values are mainly concentrated in spatial variables. Census Tract columns have the highest missing shares, with more than half of the observations missing. This is expected because the City of Chicago suppresses Census Tracts for some trips for privacy reasons. Therefore, Census Tracts are not suitable as the main spatial aggregation level.

Community Areas are much more complete. Since pickup community area is missing in only 2.79% of the observations, it is a suitable spatial unit for the baseline demand aggregation.

Pickup centroid coordinates are also mostly complete, with around 2.74% missing values. These coordinates can be used to construct H3 or hexagon-based spatial units. However, they should be interpreted as approximate area centroids, not exact pickup locations.

Dropoff variables have more missing values than pickup variables. Since the demand model focuses on where trips start, dropoff missing values should not be used to remove observations from the main demand dataset.

In [ ]:
# missing values focused on pickup data
spatial_missing = missing_overview.loc[
    [
        "pickup_census_tract",
        "pickup_community_area",
        "pickup_centroid_latitude",
        "pickup_centroid_longitude"
    ]
]

spatial_missing

,missing_count,missing_percent
pickup_census_tract,8237431,55.647605
pickup_community_area,413385,2.792604
pickup_centroid_latitude,405819,2.741493
pickup_centroid_longitude,405819,2.741493


`pickup_census_tract` has by far the highest missing share (55.65%), so Census Tracts are not suitable as the main spatial unit. `pickup_community_area` (2.79%) and pickup coordinates (2.74%) are much more complete and are therefore more reliable for the main demand aggregation.

In [11]:
# Missingness flag for baseline spatial unit
missing_pickup_area = df_full["pickup_community_area"].isna()

# Helper function
def missing_rate_by(group_col, missing_flag):
    return (
        df_full
        .assign(missing=missing_flag)
        .groupby(group_col)["missing"]
        .agg(["count", "sum", "mean"])
        .rename(columns={
            "count": "total_trips",
            "sum": "missing_count",
            "mean": "missing_percent"
        })
        .assign(missing_percent=lambda x: x["missing_percent"] * 100)
        .sort_values("missing_percent", ascending=False)
    )

In [12]:
missing_area_by_company = missing_rate_by("company", missing_pickup_area)

missing_area_by_company.head(15)

,total_trips,missing_count,missing_percent
company,,,
4623 - 27290 Jay Kim,2156,2156,100.000000
4787 - 56058 Reny Cab Co,83,83,100.000000
Tac - American United Non Dispatch,21,11,52.380952
Tac - American United Dispatch,18031,5679,31.495757
Blue Ribbon Taxi Association Inc.,21,4,19.047619
Tac - Blue Diamond Non Dispatch,309,52,16.828479
Chicago City Taxi Association,128820,21355,16.577395
Tac - Blue Diamond Dispatch,5701,825,14.471145
Tac - Yellow Non Color,5748,769,13.378566


The highest missing rates occur mostly for small companies, where percentages are unstable because they are based on few trips. Some larger companies also show above-average missing rates, but the affected rows still represent only a small share of the full dataset. Therefore, we do not exclude companies entirely. For the baseline Community Area dataset, only rows with missing `pickup_community_area` are removed later.

In [13]:
missing_area_by_hour = (
    df_full
    .assign(
        hour=df_full["trip_start_timestamp"].dt.hour,
        missing=missing_pickup_area
    )
    .groupby("hour")["missing"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "total_trips",
        "sum": "missing_count",
        "mean": "missing_percent"
    })
)

missing_area_by_hour["missing_percent"] *= 100

missing_area_by_hour

,total_trips,missing_count,missing_percent
hour,,,
0,234351,8825,3.765719
1,137321,6054,4.408648
2,80887,3688,4.559447
3,62952,2969,4.716292
4,81488,5583,6.851316
5,143511,7357,5.126436
6,281914,9034,3.204523
7,544657,22000,4.039239
8,798324,28151,3.526263


Missing `pickup_community_area` values vary slightly by hour, with somewhat higher rates during the early morning hours. However, the differences are moderate and there is no extreme hourly concentration. Therefore, the missing values do not appear to strongly distort the temporal demand pattern.

In [14]:
missing_area_by_month = (
    df_full
    .assign(
        month=df_full["trip_start_timestamp"].dt.month,
        missing=missing_pickup_area
    )
    .groupby("month")["missing"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "total_trips",
        "sum": "missing_count",
        "mean": "missing_percent"
    })
)

missing_area_by_month["missing_percent"] *= 100

missing_area_by_month

,total_trips,missing_count,missing_percent
month,,,
1,1313121,39557,3.012441
2,1340039,39418,2.941556
3,1670264,45684,2.735136
4,1124241,32366,2.878920
5,1271845,35031,2.754345
6,1254541,31187,2.485929
7,1135646,29893,2.632246
8,1149340,30724,2.673186
9,1186885,33938,2.859418


Missing `pickup_community_area` rates are very similar across months, ranging from about 2.49% to 3.01%. Therefore, removing these rows is unlikely to skew the monthly distribution of trips.

In [15]:
trip_characteristics_by_missing_area = (
    df_full
    .assign(missing_pickup_area=missing_pickup_area)
    .groupby("missing_pickup_area")[["trip_seconds", "trip_miles", "trip_total"]]
    .agg(["count", "mean", "median"])
)

trip_characteristics_by_missing_area

trip_seconds                      trip_miles            \
                           count         mean  median      count      mean   
missing_pickup_area                                                          
False                   14386873  3419.749384  2630.0   14389352   6.53513   
True                      413129  2297.570735  1282.0     413358  6.478296   

                           trip_total                    
                    median      count       mean median  
missing_pickup_area                                      
False                 3.04   14357822  26.935565  17.47  
True                   1.6     412084  36.428352   29.0

Trips with missing `pickup_community_area` differ somewhat from complete trips: they have shorter durations and lower median distances, but a higher average and median `trip_total`. However, since only 2.79% of trips are affected, removing them for the Community Area dataset is acceptable and unlikely to strongly distort the overall demand counts.

In [16]:
# Quantify how many rows remain usable per planned aggregation dataset

required_cols = {
    "Community Area × time": ["pickup_community_area", "trip_start_timestamp"],
    "H3 / hexagon × time":   ["pickup_centroid_latitude", "pickup_centroid_longitude", "trip_start_timestamp"],
    "Census Tract × time":   ["pickup_census_tract", "trip_start_timestamp"],
}

n_total = len(df_full)

usability = pd.DataFrame(
    [
        {
            "dataset": name,
            "required_columns": ", ".join(cols),
            "usable_rows": df_full[cols].dropna().shape[0],
            "usable_percent": df_full[cols].dropna().shape[0] / n_total * 100,
            "lost_percent": (1 - df_full[cols].dropna().shape[0] / n_total) * 100,
        }
        for name, cols in required_cols.items()
    ]
)

usability

,dataset,required_columns,usable_rows,usable_percent,lost_percent
0,Community Area × time,"pickup_community_area, trip_start_timestamp",14389464,97.207396,2.792604
1,H3 / hexagon × time,"pickup_centroid_latitude, pickup_centroid_long...",14397030,97.258507,2.741493
2,Census Tract × time,"pickup_census_tract, trip_start_timestamp",6565418,44.352395,55.647605


The table makes the consequence of the missing value strategy explicit. The Community Area and H3 datasets retain almost all trips and are therefore suitable as main aggregation levels. The Census Tract dataset loses more than half of all trips, which is why it is only used as a sensitivity check and is not used to estimate absolute demand levels.

The actual row removal is not performed here. It is applied later in section 1.6, where each aggregation dataset is built from the cleaned trip-level data using its own filter.


In [33]:
pd.crosstab(
    df_full["pickup_community_area"].isna(),
    df_full["pickup_centroid_latitude"].isna(),
    margins=True
)

pickup_centroid_latitude,False,True,All
pickup_community_area,,,
False,14389464,0,14389464
True,7566,405819,413385
All,14397030,405819,14802849


The missingness in pickup_community_area and pickup_centroid_latitude is almost entirely co-occurring: only 7,566 rows (0.05%) have one but not the other. Both columns are therefore missing for the same underlying reason and the Community Area and H3 datasets will be based on nearly identical row sets.

No rows are dropped at this stage. df_full remains unchanged throughout section 1.3. The actual filtering is applied in section 1.6, separately for each aggregation dataset, using only the columns required for the respective spatial resolution.

## 1.4 Semantic Validation and Outlier Handling

After analyzing missing values, the next step is to check whether the available values are meaningful from a business and domain perspective. While syntactical cleaning ensures that columns have the correct format, semantic validation checks whether the recorded trips are plausible taxi trips.

This step builds on the missing value strategy from the previous section. Semantic checks are only performed where the required values are available. Missing values are therefore not removed globally, but handled depending on the specific validation or later aggregation task.

This step is important because implausible values can distort demand aggregation and model training. For example, trips with negative duration, negative distance, unrealistic timestamps, or extreme trip totals may not represent valid demand observations.

The main checks in this section include:

- validating the chronological order of trip start and trip end timestamps, where trip end needs to be after trip start
- checking trip duration values, especially zero, negative, and extremely long trips
- checking trip distance values, especially zero, negative, and extremely long trips
- identifying suspicious combinations of distance and price, while avoiding automatic exclusion because short but expensive trips may still be possible due to congestion, waiting time, or additional charges
- checking monetary values such as `trip_total` for negative or implausibly high amounts
- comparing `trip_seconds` with the duration calculated from start and end timestamps
- checking whether pickup coordinates are located within a plausible geographic range for Chicago
- checking whether Census Tracts are consistently linked to Community Areas, since Census Tracts are smaller spatial units and inconsistent mappings may indicate data quality issues
- analyzing companies with very few entries or strongly deviating trip characteristics
- checking for unusual or inconsistent payment types
- documenting which records should be excluded from later demand aggregation and why

The goal is not to make the dataset artificially clean, but to separate technically valid and analytically useful demand observations from records that are likely erroneous or not suitable for the intended demand modeling task. Extreme values are therefore first inspected and documented before any exclusion decision is made.

In [17]:
#Check chronological order of timestamps
invalid_order = df_full["trip_end_timestamp"] < df_full["trip_start_timestamp"]

n_invalid = invalid_order.sum()
pct_invalid = invalid_order.mean() * 100

print(f"Rows where end < start: {n_invalid} ({pct_invalid:.4f}%)")


Rows where end < start: 98 (0.0007%)


In [18]:
# Flag rows where trip end is before trip start: clear recording error
# These rows are excluded in all later aggregation steps
end_before_start = df_full["trip_end_timestamp"] < df_full["trip_start_timestamp"]

df_full.loc[end_before_start, [
    "trip_start_timestamp", "trip_end_timestamp", 
    "trip_seconds", "trip_miles", "trip_total"]]

,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,trip_total
464409,2026-03-08 00:45:00,2026-03-06 07:45:00,<NA>,38.4,121.35
730653,2026-02-20 05:30:00,2026-02-17 08:15:00,<NA>,1.8,9.75
822429,2026-02-14 00:00:00,2026-02-11 09:15:00,<NA>,22.5,56.0
1627737,2025-12-21 02:45:00,2025-12-20 13:30:00,<NA>,2.4,12.0
2067908,2025-11-29 20:15:00,2025-11-05 21:00:00,<NA>,15.1,49.45
...,...,...,...,...,...
13886367,2024-03-04 18:00:00,2024-03-04 15:45:00,<NA>,12.1,43.25
14319785,2024-02-05 11:30:00,2024-02-05 11:15:00,<NA>,0.0,7.75
14675961,2024-01-11 09:00:00,2024-01-11 08:00:00,<NA>,1.6,11.75
14677062,2024-01-11 08:00:00,2024-01-11 07:00:00,<NA>,1.4,10.0


All 98 affected rows have trip_seconds = NaN and show timestamp differences ranging from one hour to several days. This is a consistent pattern indicating systematic recording errors, not short valid trips. 

In [19]:
# Distribution of trip_seconds and edge case counts
print(df_full["trip_seconds"].describe(percentiles=[.25, .5, .75, .95, .99, .999]))
print()
print(f"Zero (= 0): {(df_full['trip_seconds'] == 0).sum()}")
print(f"Negative (< 0): {(df_full['trip_seconds'] < 0).sum()}")
print(f"> 2h (7 200 s): {(df_full['trip_seconds'] > 7_200).sum()}")
print(f"> 3h (10 800 s): {(df_full['trip_seconds'] > 10_800).sum()}")

count     14800002.0
mean     3388.424757
std      2991.053106
min              0.0
25%           1199.0
50%           2578.0
75%           5400.0
95%           8820.0
99%           9710.0
99.9%         9990.0
max          86396.0
Name: trip_seconds, dtype: Float64

Zero (= 0): 201927
Negative (< 0): 0
> 2h (7 200 s): 1895855
> 3h (10 800 s): 12104


In [20]:
#Zero or negative trip_seconds: clear recording errors
invalid_seconds = df_full["trip_seconds"] <= 0

# Extremely long trips (> 3 hours): inspect before deciding on exclusion
extreme_seconds = df_full["trip_seconds"] > 10_800

print(f"invalid_seconds (≤ 0): {invalid_seconds.sum()} rows ({invalid_seconds.mean()*100:.4f}%)")
print(f"extreme_seconds (> 3h): {extreme_seconds.sum()} rows ({extreme_seconds.mean()*100:.4f}%)")

# Inspect extreme trips
df_full.loc[extreme_seconds, ["trip_start_timestamp", "trip_seconds", "trip_miles", "trip_total"]].describe()

invalid_seconds (≤ 0): 201927 rows (1.3644%)
extreme_seconds (> 3h): 12104 rows (0.0818%)


,trip_start_timestamp,trip_seconds,trip_miles,trip_total
count,12104,12104.0,12097.0,12036.0
mean,2025-02-12 18:15:24.314276,36106.062541,19.36577,67.973028
min,2024-01-01 03:00:00,10801.0,0.0,0.0
25%,2024-08-03 16:26:15,14662.75,1.29,10.75
50%,2025-02-16 14:15:00,35301.5,9.69,37.25
75%,2025-08-20 06:18:45,52999.25,20.54,82.25
max,2026-03-31 23:00:00,86396.0,786.33,985.0
std,NaN,21114.531822,38.394907,103.671455


No negatives exist. There are Zero-second trips (201,927, ~1.36%). The distribution is clean up to the 99.9th percentile; above 3 hours only 12,104 rows (0.08%) remain, with a median of ~9.8 h, a max distance of 786 miles, and a max trip_total of $985 --> most likely not real taxi trips. These are captured in extreme_seconds and potentially excluded in section 1.6.

In [21]:
# Trip Distance (trip_miles)
print(df_full["trip_miles"].describe(percentiles=[.25, .5, .75, .95, .99, .999]))
print()
print(f"Zero (= 0): {(df_full['trip_miles'] == 0).sum()}")
print(f"Negative (< 0): {(df_full['trip_miles'] < 0).sum()}")
print(f"> 50 miles: {(df_full['trip_miles'] > 50).sum()}")
print(f"> 100 miles: {(df_full['trip_miles'] > 100).sum()}")

zero_miles = df_full["trip_miles"] == 0
extreme_miles = df_full["trip_miles"] > 100

print()
print(f"zero_miles: {zero_miles.sum()} rows ({zero_miles.mean()*100:.2f}%)")
print(f"extreme_miles (> 100 mi): {extreme_miles.sum()} rows ({extreme_miles.mean()*100:.4f}%)")

count    14802710.0
mean       6.533543
std        7.240746
min             0.0
25%            1.07
50%            3.02
75%            11.6
95%           18.31
99%           26.73
99.9%         42.48
max          979.17
Name: trip_miles, dtype: Float64

Zero (= 0): 1294820
Negative (< 0): 0
> 50 miles: 7187
> 100 miles: 1202

zero_miles: 1294820 rows (8.75%)
extreme_miles (> 100 mi): 1202 rows (0.0081%)


No negative distances exist. Zero-mile trips (1,294,820, ~8.75%) likely reflect metering issues or canceled trips; they still count as pickup demand events but are excluded from any distance- or speed-based feature calculations. Trips above 100 miles (1,202 rows, max 979 miles) are recording errors consistent with the extreme_seconds group and are captured in extreme_miles for exclusion in section 1.6.

In [22]:
# Cross-check: zero-mile trips: are trip_seconds also low or is there a discrepancy?
mask = df_full["trip_miles"] == 0
print("trip_seconds for zero-mile trips:")
print(df_full.loc[mask, "trip_seconds"].describe(percentiles=[.5, .75, .95, .99]))
print()
zero_miles_long = mask & (df_full["trip_seconds"] > 1_200)
print(f"zero miles BUT > 20 min duration: {zero_miles_long.sum()} rows ({zero_miles_long.mean()*100:.2f}%)")

trip_seconds for zero-mile trips:
count      1293385.0
mean     1521.194978
std       2889.04301
min              0.0
50%            160.0
75%           1800.0
95%           7570.0
99%           9560.0
max          86396.0
Name: trip_seconds, dtype: Float64

zero miles BUT > 20 min duration: 373910 rows (2.53%)


The median trip_seconds for zero-mile trips is 160 s, consistent with short or aborted trips. However, 373,910 rows (2.53%) show zero miles with more than 20 min duration, and the 75th percentile falls at exactly 1,800 s (30 min), suggesting capped or default values rather than real trips.

In [23]:
# suspicious distance/price combinations
print(f"Negative trip_total: {(df_full['trip_total'] < 0).sum()}")
print(f"Zero trip_total: {(df_full['trip_total'] == 0).sum()}")
print()

# Price per mile for trips with measurable distance
ppm = (
    df_full.loc[df_full["trip_miles"] > 0, "trip_total"]
    / df_full.loc[df_full["trip_miles"] > 0, "trip_miles"]
)
print("Price per mile (trip_miles > 0):")
print(ppm.describe(percentiles=[.5, .75, .95, .99]))
print()

# Short trips (< 0.5 miles, > 0) with very high total (> $50)
short_expensive = (
    (df_full["trip_miles"] > 0)
    & (df_full["trip_miles"] < 0.5)
    & (df_full["trip_total"] > 50)
)
print(f"< 0.5 mi but > $50: {short_expensive.sum()} rows ({short_expensive.mean()*100:.3f}%)")

Negative trip_total: 0
Zero trip_total: 36242

Price per mile (trip_miles > 0):
count    13479465.0
mean       17.69783
std      213.910835
min             0.0
50%        4.473684
75%        7.169118
95%            16.0
99%      108.333333
max         80000.0
dtype: Float64

< 0.5 mi but > $50: 31051 rows (0.210%)


No negatives exist for trip_total, but 36,242 zero-total trips are present. The price-per-mile distribution is strongly right-skewed: the median is $4.47/mile, which is plausible for Chicago taxis, but the 99th percentile reaches $108/mile and the maximum is $80,000/mile, driven by a small number of extreme outliers.

Short but expensive trips (< 0.5 miles, > $50) account for 31,051 rows (0.21%). These are suspicious but not automatically invalid. High extras, tolls, or waiting time could explain elevated totals for very short distances. They are therefore not excluded from demand counts but flagged for sensitivity checks in later modeling steps.

In [24]:
# Monetary values: trip_total distribution and extreme flags
print(df_full["trip_total"].describe(percentiles=[.25, .5, .75, .95, .99, .999]))
print()

# Extreme trip_total (> $200) 
extreme_total = df_full["trip_total"] > 200
print(f"extreme_total (> $200): {extreme_total.sum()} rows ({extreme_total.mean()*100:.4f}%)")
print()
df_full.loc[extreme_total, ["trip_seconds", "trip_miles", "trip_total"]].describe()

count    14769906.0
mean      27.200417
std       24.593612
min             0.0
25%            10.2
50%           17.67
75%           40.83
95%           68.14
99%            99.5
99.9%         177.1
max          999.99
Name: trip_total, dtype: Float64

extreme_total (> $200): 9541 rows (0.0646%)



,trip_seconds,trip_miles,trip_total
count,9470.0,9503.0,9541.0
mean,4989.540127,44.795591,349.410762
std,10823.134193,51.480914,178.302843
min,0.0,0.0,200.01
25%,240.0,0.0,221.5
50%,2291.0,44.55,275.75
75%,4955.0,70.075,412.0
max,86144.0,419.06,999.99


The `trip_total` distribution is clean for the bulk of trips: median $17.67, 99th percentile $99.50. The maximum is capped at exactly $999.99, which is a suspicious round value and likely a system ceiling rather than a real fare.

Trips above $200 (9,541 rows, 0.06%) have a median distance of 44.55 miles and a median duration of ~38 min — consistent with long-distance rides such as airport transfers. However, the 25th percentile shows 0 miles and very short durations, meaning a subset of high-total trips has no plausible distance recorded. The flag `extreme_total` captures this group and is excluded from revenue-based aggregations in section 1.6, but retained for demand counts.

In [25]:
# cross-check trip_seconds against timestamp difference
# Timestamps are rounded to 15-min intervals → max rounding error = 2 × 900 s = 1 800 s
ts_diff = (
    df_full["trip_end_timestamp"] - df_full["trip_start_timestamp"]
).dt.total_seconds()

discrepancy = (ts_diff - df_full["trip_seconds"]).abs()

ts_mismatch = discrepancy > 1_800

print(f"ts_mismatch (|ts_diff − trip_seconds| > 30 min): {ts_mismatch.sum()} rows ({ts_mismatch.mean()*100:.4f}%)")
print()
print("Discrepancy distribution (all rows with both values present):")
print(discrepancy.dropna().describe(percentiles=[.5, .75, .95, .99, .999]))


ts_mismatch (|ts_diff − trip_seconds| > 30 min): 7274772 rows (49.1539%)

Discrepancy distribution (all rows with both values present):
count     14800002.0
mean     2765.908126
std      2697.751252
min              0.0
50%           1782.0
75%           4800.0
95%           7940.0
99%           8760.0
99.9%         9070.0
max          85536.0
dtype: Float64


Nearly 49% of rows show a discrepancy > 30 min between `trip_seconds` and the timestamp-derived duration. However, this is primarily explained by the 15-min rounding of timestamps (max combined rounding error = 2 × 900 s = 1,800 s): the median discrepancy is 1,782 s, sitting right at this boundary. The remaining cases likely reflect that `trip_seconds` measures metered driving time, while timestamps also capture dispatch delay and waiting at pickup. This is a structural difference between the two fields, not individual recording errors. `trip_seconds` is the authoritative duration measure and is used in all calculations. The `ts_mismatch` flag is therefore not applied as an exclusion criterion. The `end_before_start` flag (98 rows) already captures the genuinely invalid cases.


In [26]:
# coordinate range check: pickup centroids within Chicago bounding box
# Chicago approx. bounds: lat [41.6, 42.1], lon [-87.95, -87.5]
lat = df_full["pickup_centroid_latitude"]
lon = df_full["pickup_centroid_longitude"]

outside_chicago = (
    lat.notna() & lon.notna() &
    ~((lat >= 41.6) & (lat <= 42.1) & (lon >= -87.95) & (lon <= -87.5))
)

print(f"outside_chicago (lat/lon present but outside bounds): {outside_chicago.sum()} rows ({outside_chicago.mean()*100:.4f}%)")
print()
print("Latitude range of flagged rows:")
print(df_full.loc[outside_chicago, "pickup_centroid_latitude"].describe())
print()
print("Longitude range of flagged rows:")
print(df_full.loc[outside_chicago, "pickup_centroid_longitude"].describe())


outside_chicago (lat/lon present but outside bounds): 0 rows (0.0000%)

Latitude range of flagged rows:
count     0.0
mean     <NA>
std      <NA>
min      <NA>
25%      <NA>
50%      <NA>
75%      <NA>
max      <NA>
Name: pickup_centroid_latitude, dtype: Float64

Longitude range of flagged rows:
count     0.0
mean     <NA>
std      <NA>
min      <NA>
25%      <NA>
50%      <NA>
75%      <NA>
max      <NA>
Name: pickup_centroid_longitude, dtype: Float64


All pickup centroids with available coordinates fall within the Chicago bounding box (lat [41.6, 42.1], lon [−87.95, −87.5]). No geographic outliers exist. The `outside_chicago` flag is not needed as an exclusion criterion.


In [27]:
# Census Tract <--> Community Area consistency
# Each Census Tract should map to exactly one Community Area
ct_ca = (
    df_full[["pickup_census_tract", "pickup_community_area"]]
    .dropna()
    .drop_duplicates()
)
tracts_multiple_areas = (
    ct_ca.groupby("pickup_census_tract")["pickup_community_area"]
    .nunique()
)
inconsistent_tracts = (tracts_multiple_areas > 1).sum()
print(f"Census Tracts mapping to >1 Community Area: {inconsistent_tracts}")
print()

# Companies: trip volume and basic characteristics
company_stats = (
    df_full.groupby("company")
    .agg(
        trip_count=("trip_id", "count"),
        median_miles=("trip_miles", "median"),
        median_seconds=("trip_seconds", "median"),
        median_total=("trip_total", "median"),
    )
    .sort_values("trip_count", ascending=False)
)
print(f"Total companies: {len(company_stats)}")
print()
print(company_stats.head(15))
print()
# Small companies (< 100 trips)
small_companies = (company_stats["trip_count"] < 100).sum()
print(f"Companies with < 100 trips: {small_companies} ({small_companies/len(company_stats)*100:.1f}% of companies)")


Census Tracts mapping to >1 Community Area: 0

Total companies: 46

                                   trip_count  median_miles  median_seconds  \
company                                                                       
Flash Cab                             3080563          5.24          2332.0   
Taxicab Insurance Agency Llc          1766271          2.69          2960.0   
Taxi Affiliation Services             1666116           2.4           264.0   
Sun Taxi                              1607690          3.03          3020.0   
City Service                          1464603          2.86          3026.0   
Chicago Independents                   938826          2.84          3065.0   
5 Star Taxi                            734321           5.9          2360.0   
Transit Administrative Center Inc      688904           2.6           348.0   
Blue Ribbon Taxi Association           627748          2.55          3150.0   
Globe Taxi                             494658          2.95    

**Census Tract consistency:** No Census Tract maps to more than one Community Area. The spatial hierarchy is fully consistent; Census Tract can be used as a sub-unit of Community Area without ambiguity.

**Companies:** 46 companies are present; 4 have fewer than 100 trips and are negligible in volume. The top 15 companies account for the vast majority of demand. Three companies stand out with unusually low median `trip_seconds` (264 s, 348 s, 516 s) compared to the typical range of ~2,300–3,200 s for the main providers. This likely reflects a different recording convention for those companies rather than genuinely shorter trips. These companies are not excluded from demand counts but their `trip_seconds` values should be treated cautiously in duration-based calculations.


In [28]:
# Payment types
payment_stats = (
    df_full.groupby("payment_type")
    .agg(
        trip_count=("trip_id", "count"),
        median_total=("trip_total", "median"),
        zero_total_pct=("trip_total", lambda x: (x == 0).mean() * 100),
    )
    .sort_values("trip_count", ascending=False)
)
payment_stats["share_pct"] = payment_stats["trip_count"] / payment_stats["trip_count"].sum() * 100
print(payment_stats)


              trip_count  median_total  zero_total_pct  share_pct
payment_type                                                     
Credit Card      5456324          37.8        0.002658  36.859959
Cash             3829937          9.75        0.861858  25.872972
Mobile           3337398          12.8         0.00039  22.545646
Prcard           1612179          28.0             0.0  10.891005
Unknown           517637          27.0         0.07824   3.496874
No Charge          40955           9.0        6.140886   0.276670
Dispute             8143          10.5        4.900516   0.055010
Prepaid              276          3.75             0.0   0.001865


Eight payment types are present. Credit Card (36.9%), Cash (25.9%), and Mobile (22.5%) together account for ~85% of all trips. Cash trips have a notably lower median total ($9.75 vs. $37.80 for Credit Card), consistent with shorter or lower-value rides rather than a data quality issue. "No Charge" (0.28%) and "Dispute" (0.055%) have elevated zero-total rates (6.1% and 4.9% respectively), indicating that these trips often result in no revenue. "Unknown" (3.5%) is likely a catch-all for unclassified payment methods. No payment type is excluded from demand counts; payment type is not a relevant filter for the demand aggregation.


In [29]:
# Exclusions summary: all boolean flags defined in section 1.4
# flags are applied as filters in section 1.6
# Note: invalid_seconds contains ONLY zero-second trips (no negatives exist in the data)

flag_defs = [
    # (name, mask, scope)
    ("end_before_start",    df_full["trip_end_timestamp"] < df_full["trip_start_timestamp"], "hard — all aggregations"),
    ("extreme_seconds",     df_full["trip_seconds"] > 10_800,                                "hard — all aggregations"),
    ("extreme_miles",       df_full["trip_miles"] > 100,                                     "hard — all aggregations"),
    ("invalid_seconds",     df_full["trip_seconds"] <= 0,                                    "soft — duration/speed features only"),
    ("zero_miles",          df_full["trip_miles"] == 0,                                      "soft — distance/speed features only"),
    ("extreme_total",       df_full["trip_total"] > 200,                                     "soft — revenue aggregations only"),
    ("missing_pickup_area", df_full["pickup_community_area"].isna(),                         "spatial — community area datasets only"),
]

summary = pd.DataFrame([
    {"flag": name, "scope": scope, "rows": int(mask.sum()), "pct": mask.mean() * 100}
    for name, mask, scope in flag_defs
])

# Combined hard exclusion mask (applied to all aggregation datasets)
hard_exclude = pd.Series(False, index=df_full.index)
for _, mask, scope in flag_defs:
    if scope.startswith("hard"):
        hard_exclude = hard_exclude | mask

n_total = len(df_full)
print(summary.to_string(index=False))
print()
print(f"Hard exclusions union:       {hard_exclude.sum():>10,}  ({hard_exclude.mean()*100:.2f}%)")
print(f"Rows after hard exclusions:  {(~hard_exclude).sum():>10,}  ({(~hard_exclude).mean()*100:.2f}%)")


               flag                                  scope    rows      pct
   end_before_start                hard — all aggregations      98 0.000662
    extreme_seconds                hard — all aggregations   12104 0.081784
      extreme_miles                hard — all aggregations    1202 0.008120
    invalid_seconds    soft — duration/speed features only  201927 1.364371
         zero_miles    soft — distance/speed features only 1294820 8.747182
      extreme_total       soft — revenue aggregations only    9541 0.064598
missing_pickup_area spatial — community area datasets only  413385 2.792604

Hard exclusions union:           13,024  (0.09%)
Rows after hard exclusions:  14,787,014  (99.91%)


Three flags define hard exclusions applied to all aggregations: `end_before_start` (98 rows, timestamp logic error), `extreme_seconds` (~12 k, > 3 h with median 9.8 h, not real taxi trips), and `extreme_miles` (~1.2 k, > 100 miles). Their union covers roughly 13 k rows (~0.09 % of the dataset). No negatives exist for either `trip_seconds` or `trip_miles`.

Three flags define soft exclusions that retain rows for demand counts but exclude them from specific feature calculations: `invalid_seconds` (~202 k, trip_seconds = 0) and `zero_miles` (~1.3 M, 8.75 %) are excluded from duration- and distance-based features respectively; `extreme_total` (~9.5 k, 0.06 %) is excluded from revenue aggregations. Zero-second and zero-mile trips still represent valid pickup events and are counted in demand.

`missing_pickup_area` (~2.8 %) is a spatial filter: rows without a Community Area assignment are excluded from community-area-level datasets but retained in all time-based and city-wide aggregations.

`ts_mismatch` and `outside_chicago` are documented but not applied as exclusions: the timestamp discrepancy is structural (15-min rounding + metered vs. dispatch time); no coordinates fall outside the Chicago bounding box.


## 1.5 Feature Engineering and External Data Enrichment

After missing values and semantic plausibility have been analyzed, additional variables are prepared to make the trip data useful for demand prediction and later fleet-related decisions. The goal is not to create as many features as possible, but to create only features that are clearly connected to the business questions.

The most important features are temporal features derived from `trip_start_timestamp`. Demand is expected to differ by hour, weekday, weekend, month, and season. Therefore, features such as `hour`, `day_of_week`, `is_weekend`, `month`, and `date` can be created. These features directly support the question of when demand occurs.

Spatial information is mainly prepared rather than newly created. Community Area and Census Tract identifiers already exist in the raw dataset and can be used later for aggregation after missing value checks. Census Tracts must be used carefully because they contain many missing values. H3 or hexagon identifiers, however, need to be newly created from pickup centroid coordinates.

Trip duration and trip distance are not direct causes of pickup demand, but they are useful for operational interpretation. Areas with the same number of pickups may still require different fleet sizes if trips differ strongly in length or duration. Therefore, features like `avg_speed_mph` are relevant for later utilization, charging, and repositioning analysis.

Monetary variables such as `trip_total`, price per mile, or price per minute are less central for the initial demand prediction task because they describe the realized trip rather than the demand event itself. They should therefore not be treated as core predictors of demand. However, features such as `price_per_mile` or `price_per_minute` may be useful later for assessing revenue potential or market attractiveness.

External data should be added with expected link to demand. Weather (`temperature`, `precipitation`, `snowfall`) should be linked to transactional rides. Regarding re-positioning especially the location of charging hubs may be really interesting to know. 

The main steps in this section include:

- creating temporal features from `trip_start_timestamp` (`hour`, `day_of_week`, `is_weekend`, `month`, `date`)
- creating H3 / hexagon identifiers from pickup centroid coordinates (`pickup_h3`)
- calculating selected operational trip-level features (`avg_speed_mph`)
- monetary features only for later business interpretation (`price_per_mile`, `price_per_minute`)
- enriching the dataset with weather data (`temperature`, `precipitation`)
- enriching the dataset with charging data

For the initial demand model, the core features are time and pickup location. Distance, duration, dropoff, monetary, and external variables are mainly used for later interpretation, fleet planning, charging strategy, and economic assessment.

**TODO:** Explain every step and code to have a complete 1.5

## 1.6 Spatial and Temporal Aggregation

After creating the relevant features, the trip-level dataset is transformed into demand observations. This is the key step where individual taxi trips are converted into demand per spatial and temporal unit.

The main target variable is `demand_count`, which represents the number of pickups within a specific location unit and time bucket. Each row in the aggregated dataset therefore describes how much demand occurred in one area during one time interval.

The aggregation can be created at different temporal and spatial resolutions.

Temporal resolutions:

- 15-minute intervals
- 1-hour intervals
- 4-hour intervals

Spatial resolutions:

- Community Area
- Census Tract
- H3 / hexagon cells based on pickup centroid coordinates

In theory, this results in nine possible aggregation datasets. However, the practical starting point is the Community Area × 1 hour dataset, because Community Area information has relatively few missing values and hourly demand is detailed enough for modeling while still being stable enough for interpretation.

The main steps in this section include:

- selecting the required pickup location column for each spatial resolution
- selecting the required time bucket for each temporal resolution
- filtering only the rows needed for the specific aggregation dataset
- grouping trips by location unit and time bucket
- calculating `demand_count` as the number of pickups
- adding aggregated trip characteristics, such as:
  - average trip distance
  - average trip duration
  - average trip total
  - number of unique taxis
- checking whether the resulting demand dataset has missing time-location combinations
- deciding whether missing combinations should be filled with zero demand
- documenting how many trips are usable for each aggregation level

For the initial modeling dataset, the first aggregation will be:

`pickup_community_area × 1 hour`

This dataset provides a clear and robust first view of where and when demand occurs. More granular datasets, such as H3 × 1 hour or Census Tract × 1 hour, can be created later for sensitivity checks or more detailed spatial analysis.

**TODO:** Explain every step and code to have a complete 1.6